# Sionna 0.19 – Main Ray Tracing Simulation
**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15

Migrated from Untitled(1).ipynb (Sionna 2.0) to Sionna 0.19.2 API.
Uses `scene.compute_paths()` and `scene.coverage_map()` (not PathSolver/RadioMapSolver).
GPS / UTM / BNG transforms, DEM lookup, TX/RX CSV loading all preserved from original.

## CELL 0 · Environment Setup & Imports

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import KDTree
from scipy import stats
from scipy.stats import spearmanr
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime

# ── TensorFlow ────────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

# ── Sionna 0.19 ───────────────────────────────────────────────────────────────
import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

# ── OFDM helpers ──────────────────────────────────────────────────────────────
_HAS_OFDM = False
try:
    from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies
    _HAS_OFDM = True
    print('OFDM    : OK  (sionna.channel)')
except (ImportError, AttributeError):
    try:
        from sionna.channel.ofdm import cir_to_ofdm_channel, subcarrier_frequencies
        _HAS_OFDM = True
        print('OFDM    : OK  (sionna.channel.ofdm)')
    except (ImportError, AttributeError):
        print('OFDM    : NOT found – power-domain fallback will be used')

# ── Mitsuba (ray casting / variant) ───────────────────────────────────────────
_HAS_MI = False
try:
    import mitsuba as mi
    try:   mi.set_variant('cuda_ad_mono_polarized')
    except: mi.set_variant('llvm_ad_mono_polarized')
    _HAS_MI = True
    print(f'Mitsuba : {mi.variant()}')
except ImportError:
    print('Mitsuba : NOT available – ray-cast ground height disabled')

# ── rasterio (DEM lookup) ──────────────────────────────────────────────────────
_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
    print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available – DEM elevation disabled')

# ── OSM / scene generation dependencies ───────────────────────────────────────
_HAS_OSM = False
try:
    import osmnx as ox
    import shapely
    _HAS_OSM = True
    print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available – install with: pip install osmnx shapely')

try:
    import pyvista as pv
    print('pyvista : OK')
except ImportError:
    print('pyvista : NOT available – install with: pip install pyvista')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')
print(f'GPU(s)  : {[g.name for g in tf.config.list_physical_devices("GPU")]}')

# ── Shared helpers ────────────────────────────────────────────────────────────
def _safe(v):
    """Extract float from TF tensor, numpy scalar, or Python number."""
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    """Convert TF tensor (real or complex-tuple) to numpy array."""
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    """Extract path_gain array (H, W) from Sionna 0.19 CoverageMap object."""
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

## CELL 0b · Install Missing OSM Dependencies

Run once if `osmnx` / `pyvista` / `open3d` are not already in the env.

In [ ]:
# Uncomment and run once to install OSM scene-generation dependencies
# import subprocess, sys
# pkgs = ['osmnx', 'pyvista', 'open3d', 'shapely', 'pyproj',
#         'ipyleaflet', 'ipyvolume', 'rasterio']
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
# print('Done – restart kernel before continuing.')

## CELL 1 · Global Configuration

All system parameters in one place. Update paths to match your project layout.

In [ ]:
# ── Project paths ─────────────────────────────────────────────────────────────
# Update these to point to your scene directory
SCENE_DIR   = '/home/georgeskai/Documents/Region/boston3602'
SCENE_XML   = os.path.join(SCENE_DIR, 'bostonflat_scene3602/scene.xml')
PARAMS_JSON = os.path.join(SCENE_DIR, 'scene_parameters.json')
TX_CSV      = os.path.join(SCENE_DIR, 'transmitter_positions.csv')
RX_CSV      = os.path.join(SCENE_DIR, 'receiver_locations.csv')
DEM_TIFF    = os.path.join(SCENE_DIR, 'uk_terrain_boston_aoi.tif')
OUT_DIR     = os.path.join(SCENE_DIR, 'results_sionna019')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Coordinate system ─────────────────────────────────────────────────────────
# UTM zone: 32630 = WGS84/UTM zone 30N (UK/Boston-UK), 32631 = zone 31N (Paris)
# 32618 = zone 18N (Boston MA USA)
UTM_EPSG = 32630   # adjust for your area
BNG_EPSG = 27700   # British National Grid (for UK DEM TIFFs in OSGB CRS)

# ── RF parameters ─────────────────────────────────────────────────────────────
FREQUENCY_HZ  = 3.6e9
BANDWIDTH_HZ  = 20e6
TX_POWER_DBM  = 43.0
TX_GAIN_DBI   = 0.0
RX_GAIN_DBI   = 0.0
LNA_GAIN_DB   = 0.0
NOISE_FLOOR   = -120.0
EIRP_DBM      = TX_POWER_DBM + TX_GAIN_DBI

# ── OFDM parameters ───────────────────────────────────────────────────────────
NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

# ── Path solver parameters (Sionna 0.19) ──────────────────────────────────────
MAX_DEPTH      = 5
NUM_SAMPLES_CM = 5_000_000
NUM_SAMPLES_PS = 2_000_000
GRID_SIZE_M    = 5.0

# ── SNR scale ─────────────────────────────────────────────────────────────────
_tx_w    = 10**((TX_POWER_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR  - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

print('=' * 60)
print('SYSTEM CONFIGURATION')
print('=' * 60)
print(f'Scene XML      : {SCENE_XML}')
print(f'TX CSV         : {TX_CSV}')
print(f'RX CSV         : {RX_CSV}')
print(f'Output dir     : {OUT_DIR}')
print(f'Frequency      : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'TX power       : {TX_POWER_DBM} dBm  →  EIRP = {EIRP_DBM} dBm')
print(f'Noise floor    : {NOISE_FLOOR} dBm')
print(f'UTM EPSG       : {UTM_EPSG}')
print(f'Max depth      : {MAX_DEPTH}')
print('=' * 60)

## CELL 2 · Scene Bounds from XML / JSON

Reads `<default>` tags from scene.xml for GPS bounding box and scene origin.
Falls back to `scene_parameters.json` if XML does not contain them.

In [ ]:
def get_xml_default(root, name, fallback=None):
    elem = root.find(f"./default[@name='{name}']")
    return float(elem.get('value')) if elem is not None else fallback

WEST = EAST = SOUTH = NORTH = center_lon = center_lat = None
XML_OK = os.path.exists(SCENE_XML)

if XML_OK:
    try:
        _root = ET.parse(SCENE_XML).getroot()
        _ver  = _root.get('version', 'unknown')
        _maj  = int(_ver.split('.')[0]) if _ver and _ver[0].isdigit() else 0
        print(f'XML version : {_ver}  →  {"✓ COMPATIBLE (Mitsuba 3)" if _maj >= 3 else "✗ INCOMPATIBLE (Mitsuba 2 – scene may fail to load)"}')
        WEST   = get_xml_default(_root, 'scenegen_min_lon')
        EAST   = get_xml_default(_root, 'scenegen_max_lon')
        SOUTH  = get_xml_default(_root, 'scenegen_min_lat')
        NORTH  = get_xml_default(_root, 'scenegen_max_lat')
        center_lon = get_xml_default(_root, 'scenegen_origin_lon')
        center_lat = get_xml_default(_root, 'scenegen_origin_lat')
    except ET.ParseError as e:
        print(f'XML parse error: {e}')
        XML_OK = False
else:
    print('✗ scene.xml NOT FOUND')

# Fallback: scene_parameters.json
if None in [WEST, EAST, SOUTH, NORTH]:
    if os.path.exists(PARAMS_JSON):
        with open(PARAMS_JSON) as f:
            sp = json.load(f)
        b = sp.get('bounds', sp.get('bbox', {}))
        WEST  = b.get('min_lon', WEST)
        EAST  = b.get('max_lon', EAST)
        SOUTH = b.get('min_lat', SOUTH)
        NORTH = b.get('max_lat', NORTH)
        print('  Using scene_parameters.json bounds as fallback.')

# Final hardcoded fallback (update for your area)
if None in [WEST, EAST, SOUTH, NORTH]:
    WEST, EAST   = -0.15072, -0.04841
    SOUTH, NORTH = 51.49784, 51.56064
    print('WARNING: No bounds found – using hardcoded fallback. Update CELL 2.')

if center_lon is None: center_lon = (WEST + EAST) / 2
if center_lat is None: center_lat = (SOUTH + NORTH) / 2

print(f'Bounds : lon [{WEST:.6f}, {EAST:.6f}]  lat [{SOUTH:.6f}, {NORTH:.6f}]')
print(f'Center : ({center_lon:.6f}, {center_lat:.6f})')

## CELL 3 · Coordinate Utilities + DEM Elevation

- `gps_to_local(lon, lat)` → UTM → subtract scene origin → local XY  
- `local_to_gps(x, y)` → reverse  
- `get_dem_elevation(local_x, local_y)` → rasterio bilinear lookup on DEM TIF  
- `ray_cast_ground_z(x, y)` → Mitsuba ray intersect for terrain height  

In [ ]:
# ── pyproj transformers ───────────────────────────────────────────────────────
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', f'EPSG:{BNG_EPSG}', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    """GPS (lon, lat) → Sionna local XY (metres from scene origin)."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Sionna local XY → GPS (lon, lat)."""
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

# ── DEM bilinear lookup ───────────────────────────────────────────────────────
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())

def get_dem_elevation(local_x, local_y):
    """Return terrain elevation (metres) at a Sionna local XY position."""
    if dem_data is None:
        return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    if _is_bng_dem:
        px, py = utm_to_bng.transform(utm_x, utm_y)
    else:
        px, py = utm_to_gps.transform(utm_x, utm_y)   # WGS84 lon/lat
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

# ── Mitsuba ray-cast ground height ────────────────────────────────────────────
def ray_cast_ground_z(x, y, max_height=2000.0):
    """Shoot a ray downward and return Z of first hit; falls back to DEM."""
    if _HAS_MI:
        try:
            ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                           mi.Vector3f(0.0, 0.0, -1.0))
            si = scene.mi_scene.ray_intersect(ray)
            if si.is_valid():
                z_val = si.p.z
                return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
        except Exception:
            pass
    return get_dem_elevation(x, y)

print('Coordinate utilities ready.')
print(f'  gps_to_local({center_lon:.4f}, {center_lat:.4f}) → {gps_to_local(center_lon, center_lat)[:2]}')
print(f'  get_dem_elevation(0, 0) → {get_dem_elevation(0, 0):.2f} m')

## CELL 3b · OSM Map Download (Optional)

Downloads buildings from OpenStreetMap using `osmnx` for the scene bounding box.
Useful for visualisation or generating a new Mitsuba scene XML.

In [ ]:
if _HAS_OSM:
    print(f'Downloading OSM buildings for [{SOUTH:.5f},{WEST:.5f},{NORTH:.5f},{EAST:.5f}] ...')
    try:
        import osmnx as ox
        # Download buildings within scene bounding box
        tags = {'building': True}
        gdf_buildings = ox.features_from_bbox(
            north=NORTH, south=SOUTH, east=EAST, west=WEST,
            tags=tags
        )
        print(f'  Downloaded {len(gdf_buildings)} building footprints')

        # Save for reference
        osm_out = os.path.join(OUT_DIR, 'osm_buildings.geojson')
        gdf_buildings.to_file(osm_out, driver='GeoJSON')
        print(f'  Saved to {osm_out}')

        # Quick plot
        fig, ax = plt.subplots(figsize=(8, 8))
        gdf_buildings.plot(ax=ax, color='steelblue', alpha=0.5, edgecolor='navy')
        ax.set_title('OSM Building Footprints')
        ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, 'osm_buildings.png'), dpi=150)
        plt.show()
    except Exception as e:
        print(f'OSM download failed: {e}')
else:
    print('osmnx not available – skip OSM download.')
    print('Install with: pip install osmnx shapely')

## CELL 4 · Load 3-D Scene & Configure Antennas

**Sionna 0.19 API:** `load_scene(path)` — no `merge_shapes` argument.
Antenna arrays configured via `scene.tx_array` / `scene.rx_array`.

In [ ]:
if not XML_OK:
    raise RuntimeError(
        'scene.xml not found or incompatible.\n'
        'Generate a Mitsuba 3 scene via sionna_web (Steps 1-3) or OSM+Blender.')

print(f'Loading scene from {SCENE_XML} ...')
# Sionna 0.19: load_scene() does NOT accept merge_shapes argument
scene = load_scene(SCENE_XML)
scene.frequency = FREQUENCY_HZ

# ── 1×1 isotropic antenna arrays ─────────────────────────────────────────────
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')
print(f'TX array      : {scene.tx_array}')
print(f'RX array      : {scene.rx_array}')

## CELL 5 · Assign ITU-R P.2040-2 Material Properties

Auto-matches scene material names by partial string.

In [ ]:
# ── ITU-R P.2040-2 @ 3.5 GHz  (eps_r, sigma [S/m], scatter_coeff, xpd_coeff) ─
_ITU_DB = {
    'concrete'          : (5.24,  0.130, 0.40, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20),
    'wood'              : (1.99,  0.005, 0.25, 0.30),
    'glass'             : (6.27,  0.012, 0.08, 0.10),
    'metal'             : (1.00,  1e7,   0.05, 0.10),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05),
    'water'             : (81.0,  0.500, 0.02, 0.05),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20),
    'marble'            : (7.07,  0.020, 0.08, 0.10),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

print('=' * 70)
print('ASSIGNING ITU-R MATERIAL PROPERTIES  (auto-match by name)')
print('=' * 70)
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd = _ITU_DB.get(key, _DEFAULT_MAT)
    try: mat.relative_permittivity = eps_r
    except Exception: pass
    try: mat.conductivity = sigma
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, S); break
            except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  eps={eps_r:.2f}  σ={sigma:.4g}  S={S:.2f}')
print('Done.')

## CELL 6 · Load Transmitter (GPS → local XY, ray-cast Z)

Reads `transmitter_positions.csv` columns: `name, lon, lat, height, power_dbm`.
Falls back to project.json if CSV absent.

In [ ]:
print('=' * 70)
print('CELL 6 – LOAD TRANSMITTER')
print('=' * 70)

for nm in list(scene.transmitters.keys()):
    scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', EIRP_DBM))
    source   = 'CSV'
else:
    tx_name  = 'tx0'
    tx_lon, tx_lat = center_lon, center_lat
    tx_agl   = 25.0
    tx_power = EIRP_DBM
    source   = 'default (no CSV found)'
    print(f'  TX CSV not found – using {source}')

print(f'[1] Source      : {source}')
print(f'    GPS         : ({tx_lon:.6f}, {tx_lat:.6f})  AGL={tx_agl:.1f} m')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
print(f'[2] Local XY    : ({local_x:.2f}, {local_y:.2f})')

ground_z = ray_cast_ground_z(local_x, local_y)
abs_z    = ground_z + tx_agl
print(f'[3] Ground Z    : {ground_z:.2f} m  +  AGL {tx_agl:.1f} m  →  abs Z={abs_z:.2f} m')

tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)),
                 power_dbm=float(tx_power))
scene.add(tx)
print(f'[4] ✓ Added TX "{tx_name}"  pos=({local_x:.1f}, {local_y:.1f}, {abs_z:.1f})  EIRP={tx_power:.1f} dBm')

## CELL 7 · Load Receivers (GPS → local XY, ray-cast Z)

Reads `receiver_locations.csv` columns: `name, lon, lat, height`.
Converts each GPS → UTM → local XY and ray-casts ground Z per receiver.

In [ ]:
print('=' * 70)
print('CELL 7 – LOAD RECEIVERS')
print('=' * 70)

for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV)
    print(f'[1] Loaded {len(df_rx)} receivers from {RX_CSV}')
    print('[2] Converting GPS → local XY + ray-cast ground Z ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon  = float(row['lon'])
        lat  = float(row['lat'])
        agl  = float(row.get('height', 1.5))
        x, y, _ = gps_to_local(lon, lat)
        gz   = ray_cast_ground_z(x, y)
        z    = gz + agl
        nm   = str(row.get('name', f'RX_{i+1:04d}'))
        rx   = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx)
        receivers.append(rx)
    print(f'    Done in {time.time()-t0:.2f} s')
else:
    print('RX CSV not found – placing a single default receiver at 100 m offset')
    rx = Receiver(name='rx0', position=(100.0, 0.0, abs_z - tx_agl + 1.5))
    scene.add(rx)
    receivers.append(rx)

print(f'[3] {len(receivers)} receivers placed')
print('[4] First 5 receivers:')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f}, {y:8.1f})  Z={z:.2f}  GPS=({lon:.5f}, {lat:.5f})')

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

## CELL 8 · Coverage Map — With & Without Scattering

**Sionna 0.19 API:** `scene.coverage_map(...)` replaces `RadioMapSolver`.
Parameters: `cm_cell_size`, `max_depth`, `num_samples`, `los`, `specular_reflection`,
`diffuse_reflection`, `refraction`, `diffraction`.

In [ ]:
print('Computing coverage maps (Sionna 0.19 API) ...')

# Compute ground Z at scene centre for radio map plane
try:
    _bbox = scene.mi_scene.bbox()
    cx = (float(_bbox.min[0]) + float(_bbox.max[0])) / 2
    cy = (float(_bbox.min[1]) + float(_bbox.max[1])) / 2
except Exception:
    cx = cy = 0.0

ground_z_centre = ray_cast_ground_z(cx, cy)
if ground_z_centre == 0.0:
    ground_z_centre = get_dem_elevation(cx, cy)
cm_height = ground_z_centre + 1.5
print(f'  Coverage map plane Z = {ground_z_centre:.2f} + 1.5 = {cm_height:.2f} m')

# ── With scattering ───────────────────────────────────────────────────────────
print('  Running: WITH scattering ...')
cm_scatter = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)

# ── Without scattering ────────────────────────────────────────────────────────
print('  Running: WITHOUT scattering ...')
cm_no_scatter = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = False,
    refraction          = False,
    diffraction         = False,
)

cm_scatter_np    = _cm_to_numpy(cm_scatter)     # [1, H, W] path-gain (linear)
cm_no_scatter_np = _cm_to_numpy(cm_no_scatter)

# Convert to RSSI (dBm)
_tx_dbm = float(tx.power_dbm) if hasattr(tx, 'power_dbm') else TX_POWER_DBM
rssi_scatter    = 10*np.log10(cm_scatter_np[0]    + 1e-30) + _tx_dbm
rssi_no_scatter = 10*np.log10(cm_no_scatter_np[0] + 1e-30) + _tx_dbm

# ── Plot ──────────────────────────────────────────────────────────────────────
try:
    _bbox = scene.mi_scene.bbox()
    gx_min, gx_max = float(_bbox.min[0]), float(_bbox.max[0])
    gy_min, gy_max = float(_bbox.min[1]), float(_bbox.max[1])
except Exception:
    gx_min = gy_min = -500.0; gx_max = gy_max = 500.0

tx_x = _safe(tx.position[0]); tx_y = _safe(tx.position[1])
vmin, vmax = -120, -40

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, data, title in [
    (axes[0], rssi_scatter,    f'With scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
    (axes[1], rssi_no_scatter, f'Without scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
]:
    im = ax.imshow(data, origin='lower', extent=[gx_min, gx_max, gy_min, gy_max],
                   cmap='jet', aspect='auto', vmin=vmin, vmax=vmax)
    ax.scatter(tx_x, tx_y, marker='*', s=300, c='gold', edgecolors='black', label='TX')
    sample_step = max(1, len(receivers)//200)
    rx_x = [_safe(rx.position[0]) for rx in receivers[::sample_step]]
    rx_y = [_safe(rx.position[1]) for rx in receivers[::sample_step]]
    ax.scatter(rx_x, rx_y, s=5, c='cyan', alpha=0.5, label='RX')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_title(title); ax.legend()
    plt.colorbar(im, ax=ax, label='RSSI (dBm)')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'coverage_map_comparison.png'), dpi=150)
plt.show()

print(f'\nWith scatter    RSSI: mean={np.nanmean(rssi_scatter):.1f}  min={np.nanmin(rssi_scatter):.1f}  max={np.nanmax(rssi_scatter):.1f} dBm')
print(f'Without scatter RSSI: mean={np.nanmean(rssi_no_scatter):.1f}  min={np.nanmin(rssi_no_scatter):.1f}  max={np.nanmax(rssi_no_scatter):.1f} dBm')

## CELL 9 · Path Computation (Sionna 0.19 API)

**Sionna 0.19 API:** `scene.compute_paths(...)` replaces `PathSolver`.
Returns a `Paths` object with `.a` (amplitudes), `.tau` (delays), `.cir()` method.

In [ ]:
print('Computing paths (Sionna 0.19 API) ...')
print(f'  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

paths = scene.compute_paths(
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_PS,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = False,
)
print('Done.')

# ── Extract amplitudes & delays ───────────────────────────────────────────────
a_np   = _to_numpy(paths.a)
tau_np = _to_numpy(paths.tau)

# Sionna 0.19 shape: [batch, tx, rx, paths, rx_ant, tx_ant] or similar
if a_np.ndim == 6: a_np = a_np[0]
print(f'Amplitudes shape : {a_np.shape}')
print(f'Delays shape     : {tau_np.shape}')

# ── Per-receiver path gain (dB) ───────────────────────────────────────────────
power    = np.sum(np.abs(a_np)**2, axis=tuple(range(1, a_np.ndim)))
pg_db    = 10 * np.log10(power + 1e-30)
rssi_rx  = pg_db + _tx_dbm

n_rx = len(pg_db)
print(f'\nPer-receiver path gain ({n_rx} RX):')
print(f'  mean={pg_db.mean():.1f} dB  min={pg_db.min():.1f} dB  max={pg_db.max():.1f} dB')

# ── OFDM channel frequency response (optional) ────────────────────────────────
if _HAS_OFDM:
    try:
        a_cir, tau_cir = paths.cir()
        h_freq = cir_to_ofdm_channel(FREQUENCIES, a_cir, tau_cir, normalize=False)
        h_np   = _to_numpy(h_freq)
        print(f'OFDM channel shape : {h_np.shape}')
    except Exception as e:
        print(f'OFDM CIR failed: {e}')

## CELL 10 · Per-Receiver Results CSV (with GPS coordinates)

In [ ]:
# ── Interpolate coverage map path-gain to RX positions via KDTree ─────────────
H, W = rssi_scatter.shape
x_centers = np.linspace(gx_min, gx_max, W)
y_centers  = np.linspace(gy_min, gy_max, H)
XX, YY = np.meshgrid(x_centers, y_centers)
tree = KDTree(np.column_stack([XX.ravel(), YY.ravel()]))

rx_coords = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
_, idx     = tree.query(rx_coords)
pg_cm_db   = rssi_scatter.ravel()[idx]   # RSSI from coverage map at each RX

# ── Build per-receiver DataFrame ─────────────────────────────────────────────
records = []
for i, rx in enumerate(receivers[:n_rx]):
    x   = _safe(rx.position[0])
    y   = _safe(rx.position[1])
    z   = _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    records.append({
        'receiver'    : rx.name,
        'lon'         : round(lon, 6),
        'lat'         : round(lat, 6),
        'x_m'         : round(x, 2),
        'y_m'         : round(y, 2),
        'z_m'         : round(z, 3),
        'rssi_cm_dbm' : round(float(pg_cm_db[i]), 2) if i < len(pg_cm_db) else float('nan'),
        'rssi_ps_dbm' : round(float(rssi_rx[i]),  2) if i < len(rssi_rx)  else float('nan'),
        'pg_db'       : round(float(pg_db[i]),     2) if i < len(pg_db)   else float('nan'),
    })

df_out = pd.DataFrame(records)
out_csv = os.path.join(OUT_DIR, 'receiver_results.csv')
df_out.to_csv(out_csv, index=False)
print(f'Saved {len(df_out)} receivers to {out_csv}')
print(df_out.head(10).to_string(index=False))

## CELL 11 · Path Loss vs Distance Analysis

In [ ]:
tx_pos2d = np.array([_safe(tx.position[0]), _safe(tx.position[1])])
rx_pos2d = df_out[['x_m', 'y_m']].values
dist_m   = np.linalg.norm(rx_pos2d - tx_pos2d, axis=1)

df_out['dist_m']    = dist_m
df_out['path_loss'] = -df_out['pg_db']

# Free-space path loss reference
_lam = C / FREQUENCY_HZ
d_ref = np.linspace(max(dist_m.min(), 10), dist_m.max(), 300)
fspl  = 20*np.log10(4*np.pi*d_ref / _lam)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dist_m, df_out['path_loss'], s=5, alpha=0.4, c='steelblue', label='Simulated')
ax.plot(d_ref, fspl, 'r--', lw=2, label='Free-space PL')
ax.set_xlabel('Distance TX→RX (m)')
ax.set_ylabel('Path Loss (dB)')
ax.set_title(f'Path Loss @ {FREQUENCY_HZ/1e9:.2f} GHz')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'path_loss_vs_distance.png'), dpi=150)
plt.show()

print('\nAll results saved to:', OUT_DIR)